# Paso 1: Extracción y Chunking de Reglamentos UdeC
Este cuaderno lee los 3 PDFs oficiales, extrae el texto limpio y lo divide en fragmentos (chunks) para inyectarlos en el sistema RAG.

In [ ]:
!pip install PyMuPDF pandas

In [ ]:
import fitz  # PyMuPDF
import os
import json
import re

ruta_pdfs = '../Deliverables/Reglamento/'
archivos = [
    'Calendario-Academico-Pregrado-2026.pdf',
    'Reglamento_General_de_Docencia_de_Pregrado.pdf',
    'Reglamento_de_Docencia_de_Pregrado-FI.pdf'
]

chunks_totales = []

for archivo in archivos:
    ruta_completa = os.path.join(ruta_pdfs, archivo)
    if not os.path.exists(ruta_completa):
        print(f'No se encontró: {archivo}')
        continue
        
    print(f'Procesando: {archivo}...')
    doc = fitz.open(ruta_completa)
    texto_documento = ""
    
    # Extraer texto de todas las páginas del PDF
    for pagina in doc:
        texto_documento += pagina.get_text() + "\n"
        
    # Limpieza básica: quitar múltiples saltos de línea inútiles
    texto_limpio = re.sub(r'\n+', '\n', texto_documento)
    
    # Chunking: Separar por párrafos (saltos de línea)
    parrafos = texto_limpio.split('\n')
    
    # Filtrar párrafos muy cortos (ej. números de página sueltos, firmas cortas)
    for p in parrafos:
        p = p.strip()
        if len(p) > 40:  # Solo guardamos chunks con información real
            chunks_totales.append({
                "fuente": archivo,
                "texto": p
            })

print(f'\n¡Proceso terminado! Se generaron {len(chunks_totales)} chunks en total.')

# Guardar la base de datos en un JSON estructurado
with open('base_conocimiento_udec.json', 'w', encoding='utf-8') as f:
    json.dump(chunks_totales, f, ensure_ascii=False, indent=4)

print('Guardado exitosamente en: base_conocimiento_udec.json')


In [ ]:
import pandas as pd

# Miremos los primeros 5 chunks para comprobar que el texto está limpio
df = pd.DataFrame(chunks_totales)
df.head()